# GPU K-Means — GEMM V3 Aligned

This notebook expresses **both** Lloyd steps as dense matrix multiplications (GEMM)
and runs them through cuBLAS (via CuPy), while keeping the experiment protocol
identical to Atomic V3 / Reduction V3 / CPU V3:

- the exact Atomic-V3 canonical initial centroids (SHA-256 recorded),
- the same assignment-change metric and convergence rules,
- the same empty-cluster policy (keep the previous centroid),
- JIT / library warm-up excluded from timing,
- the same result/summary schema.

Only the *formulation* of the two steps changes:

```text
Atomic V3    : point threads -> direct distance loop -> global atomicAdd sums/counts
Reduction V3 : point threads -> direct distance loop -> (K,D) reduction blocks
GEMM V3      : scores = Xaug @ W   (N x K)  -> argmin       <- GEMM #1 (assignment)
               S      = A^T @ Xaug (K x D+1) -> sums|counts <- GEMM #2 (update)
```

`Xaug = [X - mu | 1]` is built once and shared by both GEMMs.

### Build checklist

| # | Piece | Status |
|---|-------|--------|
| 1 | Dataset discovery / loading / canonical C0 / outputs / batch / summary | reused from Reduction V3 |
| 2 | Direct-distance Numba kernel (validation only) | reused from Atomic V3 |
| 3 | `gemm_assign` — GEMM #1 + argmin + changed count | **TODO A1–A5** |
| 4 | `gemm_update` — GEMM #2 (one-hot) + empty-cluster policy | **TODO B1–B5** |
| 5 | `run_kmeans_gemm` — timed loop, same protocol | provided |
| 6 | Correctness check (must pass before the batch) | provided |
| 7 | Cross-implementation comparison table | provided |
| 8 | Report write-up: why GEMM wins/loses at D=9, K≤17 | **you** |


In [ ]:
import os
import re
import math
import time
import json
import platform
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import numba
from numba import cuda

import cupy as cp


## Google Drive dataset layout

Expected structure:

```text
My Drive/
└── kmeans_data/
    ├── rows10000/
    ├── rows50000/
    ├── rows100000/
    ├── rows1000000/
    ├── rows5000000/
    └── rows10000000/
```

Files are expected to follow this naming pattern:

```text
synthetic_data_<rows>_rows_<cols>_cols_<clusters>_clusters.csv
```

Example:

```text
synthetic_data_10000_rows_10_cols_3_clusters.csv
```

The notebook parses the row count, CSV column count, and cluster count automatically.

`ID` is preserved separately and excluded from the K-Means feature matrix.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab.")

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/kmeans_data")
else:
    DATA_ROOT = Path("./kmeans_data")

OUT_DIR = DATA_ROOT / "results_gpu_gemm_v3_aligned"
ASSIGNMENTS_DIR = OUT_DIR / "assignments"
CENTROIDS_DIR = OUT_DIR / "centroids"
HISTORY_DIR = OUT_DIR / "history"

OUT_DIR.mkdir(parents=True, exist_ok=True)
ASSIGNMENTS_DIR.mkdir(parents=True, exist_ok=True)
CENTROIDS_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------
# RUN MODE
# -----------------------------------------------------------------
# "all"        -> final full benchmark across all discovered datasets
# "diagnostic" -> only the four representative diagnostic datasets
# "single"     -> one TARGET_ROWS / TARGET_CLUSTERS dataset
RUN_MODE = "all"

DIAGNOSTIC_CASES = [
    (10_000, 3),
    (1_000_000, 3),
    (5_000_000, 9),
    (10_000_000, 17),
]

TARGET_ROWS = 10_000
TARGET_CLUSTERS = 3
TARGET_CSV_COLS = None

# Optional row-size filter in all mode.
# None = all discovered row folders.
RUN_SIZES = None

# -----------------------------------------------------------------
# CONVERGENCE
# -----------------------------------------------------------------
# Original assignment-style centroid tolerance.
TOL = 1e-4

# Practical assignment stability:
# 1e-4 = at most 0.01% of points change clusters in an iteration.
ASSIGNMENT_CHANGE_FRACTION_TOL = 1e-4

# Require the assignment-change fraction condition for multiple
# consecutive iterations to avoid stopping on a transient dip.
ASSIGNMENT_STABLE_ITERS = 3

MAX_ITER = 400
SEED = 42

# -----------------------------------------------------------------
# GPU / GEMM
# -----------------------------------------------------------------
# Only used by the validation kernel copied from Atomic V3.
max_k = 64
max_d = 32
THREADS_PER_BLOCK = 256

# Rows per GEMM call. Bounds the N x K score matrix and the N x K one-hot
# matrix: 1M rows x 17 clusters x 8 B = ~136 MB each, instead of ~1.4 GB
# for the full 10M-row dataset.
GEMM_CHUNK_ROWS = 1_048_576

# -----------------------------------------------------------------
# RESUME / OUTPUT
# -----------------------------------------------------------------
SKIP_COMPLETED = True
CONTINUE_ON_ERROR = True

SAVE_ASSIGNMENTS = True
SAVE_INITIAL_CENTROIDS = True
SAVE_FINAL_CENTROIDS = True
SAVE_ITERATION_HISTORY = True

# Optional common initial-centroid source for CPU / Atomic / Reduction.
# Set this for the final fair comparison if a shared reference folder exists.
REFERENCE_INITIAL_CENTROIDS_DIR = DATA_ROOT / "results_gpu_atomic_v3"

# For a fair comparison, Reduction must not silently invent a different start.
REQUIRE_REFERENCE_INITIAL_CENTROIDS = True

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("RUN_MODE :", RUN_MODE)
print(
    "Assignment change tolerance:",
    ASSIGNMENT_CHANGE_FRACTION_TOL,
    f"({ASSIGNMENT_CHANGE_FRACTION_TOL * 100:.4f}%)"
)

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Could not find dataset folder: {DATA_ROOT}"
    )


In [ ]:
DATASET_PATTERN = re.compile(
    r"^synthetic_data_(\d+)_rows_(\d+)_cols_(\d+)_clusters\.csv$"
)

ROW_FOLDER_PATTERN = re.compile(r"^rows(\d+)$")


def discover_datasets(data_root):
    """
    Scan folders named exactly rows<number>, e.g.
      rows10000
      rows50000
      rows100000
      rows1000000
      rows5000000
      rows10000000

    and discover CSVs named:
      synthetic_data_<rows>_rows_<cols>_cols_<clusters>_clusters.csv
    """
    records = []

    for folder in sorted(data_root.iterdir()):
        if not folder.is_dir():
            continue

        folder_match = ROW_FOLDER_PATTERN.match(folder.name)
        if folder_match is None:
            continue

        folder_rows = int(folder_match.group(1))

        for csv_path in sorted(folder.glob("*.csv")):
            match = DATASET_PATTERN.match(csv_path.name)

            if match is None:
                continue

            filename_rows = int(match.group(1))
            csv_cols = int(match.group(2))
            clusters = int(match.group(3))

            records.append({
                "rows": filename_rows,
                "csv_cols": csv_cols,
                "clusters": clusters,
                "folder": folder.name,
                "folder_rows": folder_rows,
                "filename": csv_path.name,
                "path": str(csv_path),
                "folder_matches_filename_rows": (
                    folder_rows == filename_rows
                ),
            })

    if not records:
        raise FileNotFoundError(
            f"No matching datasets found under rows<number> folders in {data_root}"
        )

    catalog = (
        pd.DataFrame(records)
        .sort_values(["rows", "clusters", "csv_cols"])
        .reset_index(drop=True)
    )

    mismatches = catalog[
        ~catalog["folder_matches_filename_rows"]
    ]

    if len(mismatches):
        print("WARNING: folder/filename row-count mismatches:")
        display(
            mismatches[
                ["folder", "filename", "folder_rows", "rows"]
            ]
        )

    return catalog


dataset_catalog = discover_datasets(DATA_ROOT)

print(
    f"Discovered {len(dataset_catalog)} dataset files "
    f"across {dataset_catalog['folder'].nunique()} row folders."
)

display(
    dataset_catalog[
        ["folder", "rows", "csv_cols", "clusters", "filename"]
    ]
)


## Final run mode

The notebook defaults to:

```python
RUN_MODE = "all"
```

so it runs every matching synthetic dataset discovered under the `rows<number>` folders.

The final assignment-stability threshold is:

```python
ASSIGNMENT_CHANGE_FRACTION_TOL = 1e-4
ASSIGNMENT_STABLE_ITERS = 3
```

That means the clustering may stop after **three consecutive iterations in which no more than 0.01% of points change cluster**, or earlier if the centroid-movement tolerance is met.

This criterion is size-normalized, so it can be applied consistently to 10K, 1M, 5M, and 10M datasets.


In [ ]:
def select_dataset(
    catalog,
    target_rows,
    target_clusters,
    target_csv_cols=None
):
    matches = catalog[
        (catalog["rows"] == target_rows)
        & (catalog["clusters"] == target_clusters)
    ]

    if target_csv_cols is not None:
        matches = matches[
            matches["csv_cols"] == target_csv_cols
        ]

    if len(matches) == 0:
        raise ValueError(
            "No dataset matched the requested configuration.\n\n"
            + catalog[
                ["folder", "rows", "csv_cols", "clusters", "filename"]
            ].to_string(index=False)
        )

    if len(matches) > 1:
        raise ValueError(
            "More than one dataset matched. "
            "Set TARGET_CSV_COLS to disambiguate.\n\n"
            + matches.to_string(index=False)
        )

    return matches.iloc[0]


selected = None
DATASET_PATH = None

if RUN_MODE == "single":
    selected = select_dataset(
        dataset_catalog,
        TARGET_ROWS,
        TARGET_CLUSTERS,
        TARGET_CSV_COLS
    )

    DATASET_PATH = Path(selected["path"])

    print("Selected:", DATASET_PATH)
else:
    print(
        "Batch mode enabled: every matching CSV inside every "
        "rows<number> folder will be processed."
    )


In [ ]:
LABEL_NAMES = {
    "true_label", "label", "labels",
    "cluster", "clusters", "cluster_id",
    "target", "class", "y"
}

ID_NAMES = {"id", "index", "idx", "row", "row_id"}


def load_dataset_for_kmeans(path, selected_meta):
    df = pd.read_csv(path)

    id_candidates = [
        c for c in df.columns
        if str(c).strip().lower() in ID_NAMES
    ]
    id_col = id_candidates[0] if id_candidates else None

    if id_col is not None:
        ids = df[id_col].astype(str).to_numpy()
    else:
        ids = np.arange(len(df)).astype(str)

    label_candidates = [
        c for c in df.columns
        if str(c).strip().lower() in LABEL_NAMES
    ]
    label_col = label_candidates[0] if label_candidates else None

    true_label = (
        df[label_col].to_numpy()
        if label_col is not None
        else None
    )

    excluded = {id_col, label_col}

    feature_cols = [
        c for c in df.columns
        if c not in excluded
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    if not feature_cols:
        raise ValueError(
            f"No numeric feature columns found in {path.name}"
        )

    df_data = df[feature_cols]

    N = len(df_data)
    D = len(feature_cols)
    K = int(selected_meta["clusters"])

    print("\nDataset verification")
    print("--------------------")
    print("File                     :", path.name)
    print("Actual rows              :", N)
    print("Actual total CSV columns :", len(df.columns))
    print("ID column                :", id_col)
    print("Label column excluded    :", label_col)
    print("K-Means feature count D  :", D)
    print("Cluster count K          :", K)
    print("Feature columns          :", feature_cols)

    if N != int(selected_meta["rows"]):
        print("WARNING: actual row count differs from filename metadata.")

    if len(df.columns) != int(selected_meta["csv_cols"]):
        print(
            "NOTE: actual CSV column count differs from filename metadata. "
            "A generated label column may explain the difference."
        )

    return (
        df,
        ids,
        df_data,
        true_label,
        feature_cols,
        N,
        D,
        K,
        id_col,
        label_col,
    )


if RUN_MODE == "single":
    (
        df,
        ids,
        df_data,
        true_label,
        d_dims,
        N1,
        D1,
        K1,
        ID_COLUMN_FOUND,
        LABEL_COLUMN_FOUND,
    ) = load_dataset_for_kmeans(
        DATASET_PATH,
        selected
    )

    display(df_data.head())


## Validation kernel (from Atomic V3)

This is the direct-distance assignment kernel used by Atomic and Reduction.
GEMM V3 does **not** use it in the timed loop; the correctness check runs it once
to confirm that the GEMM assignments match the other GPU implementations exactly.


In [ ]:
@cuda.jit
def calculate_clusters(
    data,
    centroids,
    assign,
    changed_count
):
    """
    One CUDA thread processes one point and assigns it to the nearest centroid.

    This V2 kernel also counts how many assignments changed.

    To avoid a highly contended global atomic for every changed point:
      1. each thread writes 0/1 into shared memory,
      2. the block reduces those flags,
      3. only thread 0 performs ONE atomicAdd per block.

    This change-count mechanism should be reused by the reduction version
    later so Atomic-vs-Reduction timing remains fair.
    """

    i = cuda.grid(1)

    row = data.shape[0]
    dim = data.shape[1]
    clusters = centroids.shape[0]

    th_id = cuda.threadIdx.x
    block_size = cuda.blockDim.x

    # Shared centroid cache.
    shared_centroids = cuda.shared.array(
        shape=(max_k, max_d),
        dtype=np.float64
    )

    # One change flag per thread.
    # This notebook uses THREADS_PER_BLOCK = 256.
    shared_changed = cuda.shared.array(
        shape=256,
        dtype=np.int32
    )

    # ------------------------------------------------------------
    # Cooperative centroid loading
    # ------------------------------------------------------------
    idx = th_id
    total = clusters * dim

    while idx < total:
        k = idx // dim
        j = idx % dim
        shared_centroids[k, j] = centroids[k, j]
        idx += block_size

    cuda.syncthreads()

    # ------------------------------------------------------------
    # Point assignment
    # ------------------------------------------------------------
    local_changed = 0

    if i < row:
        best_cluster = 0
        best_distance = math.inf

        for k in range(clusters):
            distance = 0.0

            for j in range(dim):
                diff = data[i, j] - shared_centroids[k, j]
                distance += diff * diff

            if distance < best_distance:
                best_distance = distance
                best_cluster = k

        old_cluster = assign[i]

        if old_cluster != best_cluster:
            local_changed = 1

        assign[i] = best_cluster

    # ------------------------------------------------------------
    # Block-level reduction of assignment-change flags
    # ------------------------------------------------------------
    shared_changed[th_id] = local_changed
    cuda.syncthreads()

    stride = block_size // 2

    while stride > 0:
        if th_id < stride:
            shared_changed[th_id] += shared_changed[th_id + stride]

        cuda.syncthreads()
        stride //= 2

    # Only one global atomic per block.
    if th_id == 0 and shared_changed[0] > 0:
        cuda.atomic.add(
            changed_count,
            0,
            shared_changed[0]
        )


## GEMM #1 — assignment step

Squared distance expands into three terms:

$$
\lVert x_i - c_k\rVert^2 = \underbrace{\lVert x_i\rVert^2}_{\text{same for every }k}
\;-\; 2\,x_i\!\cdot c_k \;+\; \lVert c_k\rVert^2
$$

The first term does not change the `argmin` over $k$, so we only need

$$
\text{score}_{ik} = \lVert c_k\rVert^2 - 2\,x_i\!\cdot c_k .
$$

Using the augmented row $[x_i,\,1]$, this is **one** dot product:

$$
\text{score}_{ik} = [x_i,\,1]\cdot\begin{bmatrix}-2c_k\\ \lVert c_k\rVert^2\end{bmatrix}
\quad\Longrightarrow\quad
\underbrace{\text{Scores}}_{N\times K} = \underbrace{X_{\text{aug}}}_{N\times(D+1)}\;\underbrace{W}_{(D+1)\times K}
$$

That single product is a GEMM, which cuBLAS `dgemm` runs when you write `@` on CuPy float64 arrays.

**Precision note.** The expansion subtracts large, nearly equal numbers when $|x|$ is big
compared to the cluster spread (our data: coordinates up to ±1000, spread 1–10).
`run_kmeans_gemm` subtracts the data mean `mu` first. That changes no distance, label or
centroid shift, but it keeps the terms small. The CPU NumPy path does the same thing.


In [ ]:
def gemm_assign(d_Xaug, d_C, d_labels, chunk_rows=GEMM_CHUNK_ROWS):
    """
    Assignment step as ONE GEMM per row-chunk.

    d_Xaug   : cp.ndarray (N, D+1) float64 -- centred data + trailing column of ones
    d_C      : cp.ndarray (K, D)   float64 -- centred centroids
    d_labels : cp.ndarray (N,)     int32   -- previous labels; overwritten in place

    Returns
    -------
    changed : int -- points whose label changed (same metric as Atomic/Reduction;
                     labels start at -1, so iteration 1 reports changed == N)
    """
    N = d_Xaug.shape[0]
    K, D = d_C.shape
    W = cp.empty((D + 1, K), dtype=cp.float64)
    W[:D, :] = -2 * d_C.T
    W[D, :] = (d_C * d_C).sum(axis=1)

    changed = cp.zeros((), dtype=cp.int64)

    for s in range(0, N, chunk_rows):
        e = min(s + chunk_rows, N)
        scores = d_Xaug[s:e] @ W
        new = scores.argmin(axis=1).astype(cp.int32)
        changed += cp.count_nonzero(new != d_labels[s:e])
        d_labels[s:e] = new

    return int(changed)


## GEMM #2 — update step

Let $A \in \{0,1\}^{N\times K}$ be the one-hot assignment matrix, $A_{ik}=1$ iff point $i$ is in cluster $k$. Then

$$
\underbrace{S}_{K\times(D+1)} = A^{\top}\,X_{\text{aug}}
= \Big[\; \underbrace{\textstyle\sum_{i\in k} x_i}_{\text{centroid sums}} \;\Big|\; \underbrace{\textstyle\sum_{i\in k} 1}_{\text{counts}} \;\Big]
$$

Because of the column of ones, **sums and counts come out of one GEMM**. Atomic needs
atomics for this and Reduction needs two separate reduction kernels.

New centroid $= S_{k,:D} / S_{k,D}$ when the count is $>0$; otherwise keep the previous centroid
(same empty-cluster policy as every other implementation).

Things to think about for the write-up: $A$ is 1 nonzero per row, so the dense GEMM does
$K\times$ more multiply-adds than needed. That's the trade-off: a dense GEMM does redundant work but runs
at cuBLAS speed, while atomics/reductions do minimal work with irregular access. A sparse
`cupyx.scipy.sparse.csr_matrix` SpMM is a good optional variant to benchmark.


In [ ]:
def gemm_update(d_Xaug, d_labels, d_C, chunk_rows=GEMM_CHUNK_ROWS):
    """
    Update step as ONE GEMM per row-chunk:  S = A^T @ Xaug  -> (K, D+1)

    d_Xaug   : cp.ndarray (N, D+1) float64
    d_labels : cp.ndarray (N,)     int32
    d_C      : cp.ndarray (K, D)   float64 -- current centred centroids (for empty clusters)

    Returns
    -------
    d_C_new  : cp.ndarray (K, D) float64 -- new centred centroids
    d_counts : cp.ndarray (K,)   float64 -- points per cluster
    """
    N, D1 = d_Xaug.shape
    D = D1 - 1
    K = d_C.shape[0]

    S = cp.zeros((K, D + 1), dtype=cp.float64)
    cluster_ids = cp.arange(K, dtype=cp.int32)

    for s in range(0, N, chunk_rows):
        e = min(s + chunk_rows, N)
        A = (d_labels[s:e, None] == cluster_ids[None, :]).astype(cp.float64)   # (chunk, K)
        S += A.T @ d_Xaug[s:e]    

    sums = S[:, :D];  
    counts = S[:, D]
    d_C_new = cp.where(counts[:, None] > 0, sums / cp.maximum(counts, 1)[:, None], d_C)

    return d_C_new, counts


## Timed driver — same protocol as Atomic / Reduction V3

Timing boundaries match the other GPU notebooks:

- **not timed:** CSV load, host→device copy, centring, building `Xaug`, warm-up
  (cuBLAS handle creation, CuPy kernel compilation, memory-pool allocation),
- **timed:** every iteration's assignment, update, the `(K, D)` centroid copy to host,
  and the convergence test.

CuPy launches are asynchronous, so the timer only stops after a full device synchronize.


In [ ]:
def _sync():
    cp.cuda.Device().synchronize()


def run_kmeans_gemm(
    df_data,
    K,
    max_iter=MAX_ITER,
    tol=TOL,
    assignment_fraction_tol=ASSIGNMENT_CHANGE_FRACTION_TOL,
    assignment_stable_iters=ASSIGNMENT_STABLE_ITERS,
    initial_indices=None,
    initial_centroids=None,
    seed=SEED,
    chunk_rows=GEMM_CHUNK_ROWS,
    verbose=True
):
    """
    GEMM-based GPU K-Means using the same convergence/timing protocol
    as Atomic V3 and Reduction V3.
    """
    h_data = np.ascontiguousarray(
        np.asarray(df_data, dtype=np.float64)
    )

    N, D = h_data.shape

    if verbose:
        print(
            f"--- Running K-Means GEMM V3 "
            f"(N={N}, D={D}, K={K}, chunk_rows={chunk_rows}) ---"
        )

    # ------------------------------------------------------------
    # Initial centroids (identical logic to Reduction V3)
    # ------------------------------------------------------------
    if initial_centroids is not None:
        h_centroids = np.ascontiguousarray(
            np.asarray(initial_centroids, dtype=np.float64)
        )

        if h_centroids.shape != (K, D):
            raise ValueError(
                f"initial_centroids shape {h_centroids.shape} "
                f"does not match expected {(K, D)}"
            )

        initial_indices_used = None
    else:
        if initial_indices is None:
            rng = np.random.default_rng(seed)
            initial_indices = rng.choice(
                N,
                K,
                replace=False
            )

        initial_indices = np.asarray(
            initial_indices,
            dtype=np.int64
        )

        h_centroids = h_data[initial_indices].copy()
        initial_indices_used = initial_indices.copy()

    initial_centroids_used = h_centroids.copy()

    # ------------------------------------------------------------
    # Persistent GPU data: Xaug = [X - mu | 1]   (not timed)
    # ------------------------------------------------------------
    mu = h_data.mean(axis=0)

    d_Xaug = cp.empty((N, D + 1), dtype=cp.float64)
    d_Xaug[:, :D] = cp.asarray(h_data)
    d_Xaug[:, :D] -= cp.asarray(mu)
    d_Xaug[:, D] = 1.0

    h_C = initial_centroids_used - mu
    d_C = cp.asarray(h_C)
    d_labels = cp.full(N, -1, dtype=cp.int32)

    _sync()

    # ------------------------------------------------------------
    # Warm-up -- excluded from timing.
    # Uses a chunk-sized slice so the memory pool already holds
    # chunk-sized blocks when the timed loop starts.
    # ------------------------------------------------------------
    warm_n = min(N, chunk_rows)

    warmup_start = time.perf_counter()

    d_warm_labels = cp.full(warm_n, -1, dtype=cp.int32)
    gemm_assign(d_Xaug[:warm_n], d_C, d_warm_labels, chunk_rows)
    gemm_update(d_Xaug[:warm_n], d_warm_labels, d_C, chunk_rows)
    _sync()

    warmup_time = time.perf_counter() - warmup_start
    del d_warm_labels

    if verbose:
        print(
            f"CuPy/cuBLAS warm-up: {warmup_time:.4f} s "
            "(excluded)"
        )

    # ------------------------------------------------------------
    # Timed execution
    # ------------------------------------------------------------
    start_time = time.perf_counter()

    cent_shift = math.inf
    changed = N
    changed_fraction = 1.0

    converged = False
    convergence_reason = "max_iterations"
    stable_fraction_streak = 0
    history = []

    for iteration in range(max_iter):
        old_C = h_C

        # 1) GEMM #1: assignment + changed count (int() inside syncs).
        changed = gemm_assign(
            d_Xaug,
            d_C,
            d_labels,
            chunk_rows
        )

        changed_fraction = (
            changed / float(N)
        )

        # 2) GEMM #2: sums | counts, then mean with empty-cluster policy.
        d_C, _ = gemm_update(
            d_Xaug,
            d_labels,
            d_C,
            chunk_rows
        )

        # Same per-iteration (K, D) device->host copy as Atomic/Reduction.
        h_C = cp.asnumpy(d_C)

        cent_shift = float(
            np.max(
                np.sqrt(
                    np.sum(
                        (h_C - old_C) ** 2,
                        axis=1
                    )
                )
            )
        )

        centroid_converged = (
            cent_shift < tol
        )

        assignment_fraction_met = (
            changed_fraction
            <= assignment_fraction_tol
        )

        if assignment_fraction_met:
            stable_fraction_streak += 1
        else:
            stable_fraction_streak = 0

        assignment_stability_met = (
            stable_fraction_streak
            >= assignment_stable_iters
        )

        history.append({
            "iteration": iteration + 1,
            "assignment_changes": changed,
            "assignment_change_fraction": changed_fraction,
            "assignment_change_percent": changed_fraction * 100.0,
            "max_centroid_movement": cent_shift,
            "stable_fraction_streak": stable_fraction_streak,
            "centroid_tolerance_met": centroid_converged,
            "assignment_fraction_tolerance_met": assignment_fraction_met,
            "assignment_stability_met": assignment_stability_met,
        })

        if verbose and (
            iteration < 5
            or assignment_fraction_met
            or (iteration + 1) % 25 == 0
        ):
            print(
                f"iter={iteration + 1:4d} | "
                f"changed={changed:10d} | "
                f"changed%={changed_fraction * 100:.5f}% | "
                f"shift={cent_shift:.8f} | "
                f"stable-streak={stable_fraction_streak}"
            )

        if centroid_converged or assignment_stability_met:
            converged = True

            if centroid_converged and assignment_stability_met:
                convergence_reason = "both"
            elif assignment_stability_met:
                convergence_reason = "assignment_fraction_stable"
            else:
                convergence_reason = "centroid_tolerance"

            break

    _sync()

    exec_time = (
        time.perf_counter() - start_time
    )

    h_assign = cp.asnumpy(d_labels)
    h_centroids = h_C + mu

    # Release GPU memory before the next dataset in the batch.
    del d_Xaug, d_C, d_labels
    cp.get_default_memory_pool().free_all_blocks()

    if verbose:
        print(f"Timed execution      : {exec_time:.4f} s")
        print(f"Warm-up excluded     : {warmup_time:.4f} s")
        print(f"Iterations           : {iteration + 1}")
        print(f"Converged            : {converged}")
        print(f"Reason               : {convergence_reason}")
        print(f"Final changes        : {changed}")
        print(
            f"Final change fraction : "
            f"{changed_fraction:.8e}"
        )
        print(f"Final shift          : {cent_shift:.8f}")

    return {
        "assignments": h_assign,
        "centroids": h_centroids,
        "initial_centroids": initial_centroids_used,
        "initial_indices": initial_indices_used,
        "iterations": iteration + 1,
        "execution_time": float(exec_time),
        "warmup_time": float(warmup_time),
        "max_movement": cent_shift,
        "assignment_changes": int(changed),
        "assignment_change_fraction": float(changed_fraction),
        "stable_fraction_streak": int(stable_fraction_streak),
        "converged": converged,
        "convergence_reason": convergence_reason,
        "history": pd.DataFrame(history),
    }


## Correctness check — must pass before the batch

On a small synthetic dataset that uses the same coordinate range as the real data, this checks:

1. **GEMM #1** labels vs brute-force float64 distances (mismatches only allowed at exact ties),
2. **GEMM #1** labels vs the Atomic/Reduction Numba kernel,
3. **GEMM #2** sums/counts/centroids vs `np.bincount`, including an **empty cluster**,
4. a full 30-iteration `run_kmeans_gemm` vs a plain NumPy Lloyd reference.

A small `chunk_rows` is used so the chunk loop is exercised even on 20K rows.
Until TODO A/B are filled in, this cell raises `NotImplementedError`. That's expected.


In [ ]:
def kmeans_reference(X, C0, iters):
    """Plain float64 Lloyd's algorithm with the shared empty-cluster policy."""
    C = C0.astype(np.float64).copy()
    labels = None
    for _ in range(iters):
        labels = ((X[:, None, :] - C[None]) ** 2).sum(axis=2).argmin(axis=1)
        for k in range(len(C)):
            members = labels == k
            if members.any():
                C[k] = X[members].mean(axis=0)
    return C, labels


def gemm_correctness_check(n=20_000, d=9, true_k=7, K=17, iters=30, seed=1, chunk=4096):
    rng = np.random.default_rng(seed)
    centers = rng.uniform(-1000, 1000, (true_k, d))
    X = centers[rng.integers(0, true_k, n)] + rng.normal(0, 5, (n, d))
    C0 = X[rng.choice(n, K, replace=False)].copy()

    mu = X.mean(axis=0)
    d_Xaug = cp.empty((n, d + 1), dtype=cp.float64)
    d_Xaug[:, :d] = cp.asarray(X - mu)
    d_Xaug[:, d] = 1.0
    d_C = cp.asarray(C0 - mu)

    report = {}

    # 1) GEMM #1 vs brute force
    d_labels = cp.full(n, -1, dtype=cp.int32)
    changed = gemm_assign(d_Xaug, d_C, d_labels, chunk_rows=chunk)
    lab_gemm = cp.asnumpy(d_labels)

    D2 = ((X[:, None, :] - C0[None]) ** 2).sum(axis=2)
    lab_ref = D2.argmin(axis=1)

    assert isinstance(changed, int), f"gemm_assign must return a Python int, got {type(changed)}"
    assert changed == n, f"first assignment must report changed == N ({n}), got {changed}"

    mism = np.flatnonzero(lab_gemm != lab_ref)
    if len(mism):
        best = D2[mism, lab_ref[mism]]
        got = D2[mism, lab_gemm[mism]]
        rel_gap = (got - best) / np.maximum(best, 1e-12)
        assert rel_gap.max() < 1e-9, (
            f"{len(mism)} label mismatches that are not ties (max rel gap {rel_gap.max():.3e})"
        )
    report["assign: agreement vs brute force %"] = 100.0 * (1 - len(mism) / n)

    # second call with unchanged centroids must report 0 changes
    assert gemm_assign(d_Xaug, d_C, d_labels, chunk_rows=chunk) == 0, \
        "re-assigning with the same centroids must report 0 changes"

    # 2) GEMM #1 vs Atomic/Reduction Numba kernel
    d_assign_nb = cuda.to_device(np.full(n, -1, dtype=np.int32))
    d_changed_nb = cuda.to_device(np.zeros(1, dtype=np.int32))
    calculate_clusters[math.ceil(n / THREADS_PER_BLOCK), THREADS_PER_BLOCK](
        cuda.to_device(np.ascontiguousarray(X)),
        cuda.to_device(np.ascontiguousarray(C0)),
        d_assign_nb,
        d_changed_nb,
    )
    lab_nb = d_assign_nb.copy_to_host()
    report["assign: agreement vs Numba kernel %"] = 100.0 * float((lab_gemm == lab_nb).mean())
    assert report["assign: agreement vs Numba kernel %"] > 99.99

    # 3) GEMM #2 vs bincount, with cluster K-1 forced empty
    lab_upd = lab_ref.copy()
    lab_upd[lab_upd == K - 1] = 0
    d_C_new, d_counts = gemm_update(
        d_Xaug, cp.asarray(lab_upd.astype(np.int32)), d_C, chunk_rows=chunk
    )
    counts_ref = np.bincount(lab_upd, minlength=K)
    C_expected = C0.copy()
    for k in range(K):
        if counts_ref[k]:
            C_expected[k] = X[lab_upd == k].mean(axis=0)

    assert d_C_new.shape == (K, d), f"gemm_update returned centroids of shape {d_C_new.shape}"
    assert np.array_equal(cp.asnumpy(d_counts).astype(np.int64), counts_ref), "counts differ"
    upd_err = float(np.abs(cp.asnumpy(d_C_new) + mu - C_expected).max())
    report["update: max |centroid - reference|"] = upd_err
    assert upd_err < 1e-8, f"centroids differ: {upd_err}"
    report["update: empty cluster kept previous centroid"] = bool(
        np.allclose(cp.asnumpy(d_C_new[K - 1]) + mu, C0[K - 1])
    )
    assert report["update: empty cluster kept previous centroid"]

    # 4) Full run, convergence switched off
    res = run_kmeans_gemm(
        X, K,
        max_iter=iters, tol=-1.0, assignment_fraction_tol=-1.0,
        initial_centroids=C0, chunk_rows=chunk, verbose=False,
    )
    C_ref, lab_ref_run = kmeans_reference(X, C0, iters)
    report["run: iterations"] = res["iterations"]
    report["run: max |centroid - reference|"] = float(np.abs(res["centroids"] - C_ref).max())
    report["run: label agreement %"] = 100.0 * float((res["assignments"] == lab_ref_run).mean())
    assert res["iterations"] == iters
    assert report["run: max |centroid - reference|"] < 1e-6, report
    assert report["run: label agreement %"] > 99.9, report

    return report


gemm_correctness_check()


## Output files

To match the CPU baseline as closely as practical, each GPU run writes:

```text
results_gpu_gemm_v3_aligned/
├── environment.json
├── summary.csv
├── initial_centroids_<dataset-tag>.csv
├── final_results_<dataset-tag>.csv
├── assignments/
│   └── assignments_<dataset-tag>.csv
└── centroids/
    └── final_centroids_<dataset-tag>.csv
```

The assignment file contains:

```text
ID, cluster, distance
```

and includes `true_label` as an extra reference column when the source CSV provides it. The label is **never** used as a clustering feature.


In [ ]:
try:
    device = cuda.get_current_device()
    gpu_name = device.name

    if isinstance(gpu_name, bytes):
        gpu_name = gpu_name.decode()

    compute_capability = ".".join(
        map(str, device.compute_capability)
    )
except Exception:
    gpu_name = "unknown"
    compute_capability = "unknown"

ENV = {
    "implementation": "gpu_gemm_cupy_cublas_v3_aligned",
    "gpu_name": gpu_name,
    "compute_capability": compute_capability,
    "platform": platform.platform(),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "numba": numba.__version__,
    "cupy": cp.__version__,
    "assignment_method": "Xaug @ W (cuBLAS dgemm) + argmin",
    "update_method": "onehot(labels)^T @ Xaug (cuBLAS dgemm)",
    "gemm_chunk_rows": GEMM_CHUNK_ROWS,
    "data_mean_centred": True,
    "threads_per_block": THREADS_PER_BLOCK,
    "max_k": max_k,
    "max_d": max_d,
    "centroid_tolerance": TOL,
    "assignment_change_fraction_tolerance":
        ASSIGNMENT_CHANGE_FRACTION_TOL,
    "assignment_stable_iterations":
        ASSIGNMENT_STABLE_ITERS,
    "max_iterations": MAX_ITER,
    "seed": SEED,
    "jit_warmup_excluded_from_timing": True,
    "canonical_initial_centroids_dir": str(REFERENCE_INITIAL_CENTROIDS_DIR),
    "data_root": str(DATA_ROOT),
}

with open(
    OUT_DIR / "environment.json",
    "w"
) as fh:
    json.dump(
        ENV,
        fh,
        indent=2
    )

ENV


## Output helpers

Distances are calculated **after convergence** against each point's assigned final centroid.

For very large datasets this is done in chunks so we do not create a full `N × D` temporary matrix at once.


In [ ]:
def dataset_tag_from_path(path):
    return Path(path).stem


def calculate_assigned_distances(
    X,
    centroids,
    assignments,
    chunk_size=500_000
):
    """
    Distance from each point to its assigned final centroid.
    Chunked to control host-memory use for multi-million-row data.
    """
    X = np.asarray(X, dtype=np.float64)
    assignments = np.asarray(assignments, dtype=np.int32)

    distances = np.empty(len(X), dtype=np.float64)

    for start in range(0, len(X), chunk_size):
        end = min(start + chunk_size, len(X))

        a = assignments[start:end]
        diff = X[start:end] - centroids[a]

        distances[start:end] = np.sqrt(
            np.sum(diff * diff, axis=1)
        )

    return distances


def calculate_wcss_from_distances(distances):
    return float(
        np.sum(
            np.asarray(distances, dtype=np.float64) ** 2
        )
    )


def initial_centroid_sha256(C):
    arr = np.ascontiguousarray(
        np.asarray(C, dtype=np.float64)
    )
    return hashlib.sha256(arr.tobytes()).hexdigest()


def _read_initial_centroid_csv(path, K, D, feature_cols=None):
    """
    Read the canonical Atomic-V3 centroid file.

    The saved format is:
        cluster,col_1,...,col_D

    The 'cluster' metadata column is explicitly excluded.
    """
    df = pd.read_csv(path)

    if "cluster" in df.columns:
        df = df.drop(columns=["cluster"])

    if feature_cols is not None and all(c in df.columns for c in feature_cols):
        df = df[list(feature_cols)]
    else:
        df = df.select_dtypes(include=[np.number])

    arr = df.to_numpy(dtype=np.float64)

    if arr.shape != (K, D):
        raise ValueError(
            f"{path} has centroid shape {arr.shape}; expected {(K, D)}"
        )

    return np.ascontiguousarray(arr, dtype=np.float64)


def find_reference_initial_centroids(tag, K, D, feature_cols=None):
    """
    Reuse canonical initial_centroids_<tag>.csv when available.

    Atomic may create the file on the first run. CPU and Reduction use
    their aligned notebooks with REQUIRE_REFERENCE_INITIAL_CENTROIDS=True.
    """
    if REFERENCE_INITIAL_CENTROIDS_DIR is None:
        return None

    ref_dir = Path(REFERENCE_INITIAL_CENTROIDS_DIR)
    path = ref_dir / f"initial_centroids_{tag}.csv"

    if not path.exists():
        if REQUIRE_REFERENCE_INITIAL_CENTROIDS:
            raise FileNotFoundError(
                f"Canonical initial-centroid file not found: {path}"
            )
        print(
            "Canonical initial-centroid file does not exist yet; "
            "Atomic will create it deterministically:",
            path
        )
        return None

    arr = _read_initial_centroid_csv(
        path,
        K,
        D,
        feature_cols=feature_cols
    )

    print("Using canonical initial centroids:", path)
    print("Initial centroid SHA256:", initial_centroid_sha256(arr))

    return arr


def save_run_outputs(
    path,
    meta,
    ids,
    feature_cols,
    true_label,
    X,
    result
):
    tag = dataset_tag_from_path(path)

    assignments = result["assignments"]
    centroids = result["centroids"]

    # ------------------------------------------------------------
    # Distances + WCSS
    # ------------------------------------------------------------
    distances = calculate_assigned_distances(
        X,
        centroids,
        assignments
    )

    wcss = calculate_wcss_from_distances(distances)

    cluster_sizes = np.bincount(
        assignments,
        minlength=len(centroids)
    )

    # ------------------------------------------------------------
    # Initial centroids
    # ------------------------------------------------------------
    if SAVE_INITIAL_CENTROIDS:
        init_df = pd.DataFrame(
            result["initial_centroids"],
            columns=feature_cols
        )
        init_df.insert(
            0,
            "cluster",
            np.arange(len(init_df))
        )
        init_df.to_csv(
            OUT_DIR / f"initial_centroids_{tag}.csv",
            index=False
        )

    # ------------------------------------------------------------
    # Final centroid table
    # ------------------------------------------------------------
    if SAVE_FINAL_CENTROIDS:
        cent_df = pd.DataFrame(
            centroids,
            columns=feature_cols
        )
        cent_df.insert(
            0,
            "cluster",
            np.arange(len(cent_df))
        )
        cent_df["cluster_size"] = cluster_sizes

        cent_df.to_csv(
            CENTROIDS_DIR / f"final_centroids_{tag}.csv",
            index=False
        )

    # ------------------------------------------------------------
    # Per-ID assignments
    # ------------------------------------------------------------
    assignment_path = ASSIGNMENTS_DIR / f"assignments_{tag}.csv"

    if SAVE_ASSIGNMENTS:
        out = pd.DataFrame({
            "ID": ids,
            "cluster": assignments,
            "distance": distances,
        })

        if true_label is not None:
            out["true_label"] = true_label

        out.to_csv(
            assignment_path,
            index=False
        )

    # ------------------------------------------------------------
    # One-row final result
    # ------------------------------------------------------------
    # ------------------------------------------------------------
    # Iteration history
    # ------------------------------------------------------------
    if SAVE_ITERATION_HISTORY:
        result["history"].to_csv(
            HISTORY_DIR / f"history_{tag}.csv",
            index=False
        )

    # ------------------------------------------------------------
    # One-row final result
    # ------------------------------------------------------------
    final_row = {
        "Implementation": "gpu_gemm_v3_aligned",
        "Initial centroid source": str(REFERENCE_INITIAL_CENTROIDS_DIR),
        "Initial centroid SHA256": initial_centroid_sha256(result["initial_centroids"]),
        "Dataset": tag,
        "Rows": len(X),
        "Features": X.shape[1],
        "K": len(centroids),
        "Execution time": result["execution_time"],
        "Warmup time": result["warmup_time"],
        "Warmup excluded": True,
        "Iterations": result["iterations"],
        "Per iteration ms": (
            result["execution_time"]
            / result["iterations"]
            * 1000.0
        ),
        "Converged": result["converged"],
        "Convergence reason": result["convergence_reason"],
        "Final assignment changes":
            result["assignment_changes"],
        "Final assignment change fraction":
            result["assignment_change_fraction"],
        "Final assignment change percent":
            result["assignment_change_fraction"] * 100.0,
        "Stable fraction streak":
            result["stable_fraction_streak"],
        "Max movement": result["max_movement"],
        "WCSS": wcss,
        "Cluster sizes": json.dumps(
            cluster_sizes.tolist()
        ),
        "Centroids": json.dumps(
            centroids.astype(float).tolist()
        ),
    }

    pd.DataFrame([final_row]).to_csv(
        OUT_DIR / f"final_results_{tag}.csv",
        index=False
    )

    return final_row, assignment_path


## Run / resume batch

With `RUN_MODE = "all"` the notebook now runs **every matching CSV** inside every folder named `rows<number>`.

`summary.csv` is saved **after every successful dataset**, so progress survives a Colab disconnect.


In [ ]:
SUMMARY_PATH = OUT_DIR / "summary.csv"


def update_summary_csv(new_rows):
    valid_rows = [
        row for row in new_rows
        if row is not None
    ]

    new_df = pd.DataFrame(
        valid_rows
    )

    if SUMMARY_PATH.exists():
        old_df = pd.read_csv(
            SUMMARY_PATH
        )

        if len(new_df):
            combined = pd.concat(
                [old_df, new_df],
                ignore_index=True
            )
        else:
            combined = old_df
    else:
        combined = new_df

    if len(combined):
        combined = combined.drop_duplicates(
            subset=[
                "Dataset",
                "Implementation"
            ],
            keep="last"
        )

        combined = combined.sort_values(
            [
                "Rows",
                "K",
                "Implementation"
            ]
        ).reset_index(drop=True)

    combined.to_csv(
        SUMMARY_PATH,
        index=False
    )

    return combined


def final_result_path_for(path):
    tag = dataset_tag_from_path(
        path
    )

    return (
        OUT_DIR
        / f"final_results_{tag}.csv"
    )


def run_one_dataset(path, meta):
    final_path = final_result_path_for(
        path
    )

    if SKIP_COMPLETED and final_path.exists():
        print(
            "SKIP (already completed):",
            path.name
        )

        return (
            pd.read_csv(final_path)
            .iloc[0]
            .to_dict()
        )

    (
        df_full,
        ids_local,
        df_features,
        true_label_local,
        feature_cols,
        N,
        D,
        K,
        id_col,
        label_col,
    ) = load_dataset_for_kmeans(
        path,
        meta
    )

    if K > max_k:
        raise ValueError(
            f"K={K} exceeds max_k={max_k}"
        )

    if D > max_d:
        raise ValueError(
            f"D={D} exceeds max_d={max_d}"
        )

    tag = dataset_tag_from_path(
        path
    )

    reference_C0 = (
        find_reference_initial_centroids(
            tag,
            K,
            D,
            feature_cols=feature_cols
        )
    )

    result = run_kmeans_gemm(
        df_data=df_features,
        K=K,
        max_iter=MAX_ITER,
        tol=TOL,
        seed=SEED,
        initial_centroids=reference_C0,
        verbose=True
    )

    final_row, assignment_path = (
        save_run_outputs(
            path=path,
            meta=meta,
            ids=ids_local,
            feature_cols=feature_cols,
            true_label=true_label_local,
            X=df_features.to_numpy(
                dtype=np.float64
            ),
            result=result
        )
    )

    print(
        "Saved final result:",
        final_path
    )

    if SAVE_ASSIGNMENTS:
        print(
            "Saved assignments:",
            assignment_path
        )

    if SAVE_ITERATION_HISTORY:
        print(
            "Saved history:",
            HISTORY_DIR
            / f"history_{tag}.csv"
        )

    return final_row


In [ ]:
def select_diagnostic_catalog(
    catalog,
    cases
):
    selected_rows = []

    for rows, k in cases:
        matches = catalog[
            (catalog["rows"] == rows)
            & (catalog["clusters"] == k)
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected exactly one dataset for "
                f"rows={rows}, K={k}; found {len(matches)}"
            )

        selected_rows.append(
            matches.iloc[0]
        )

    return pd.DataFrame(
        selected_rows
    ).reset_index(drop=True)


if RUN_MODE == "single":
    catalog_to_run = pd.DataFrame([
        selected
    ])

elif RUN_MODE == "diagnostic":
    catalog_to_run = (
        select_diagnostic_catalog(
            dataset_catalog,
            DIAGNOSTIC_CASES
        )
    )

elif RUN_MODE == "all":
    catalog_to_run = (
        dataset_catalog.copy()
    )

    if RUN_SIZES is not None:
        catalog_to_run = (
            catalog_to_run[
                catalog_to_run["rows"]
                .isin(RUN_SIZES)
            ].copy()
        )

else:
    raise ValueError(
        "RUN_MODE must be one of: "
        "'diagnostic', 'single', 'all'"
    )


print(
    f"Run plan: {len(catalog_to_run)} CSV files"
)

display(
    catalog_to_run[
        [
            "folder",
            "rows",
            "csv_cols",
            "clusters",
            "filename"
        ]
    ]
)

failures = []

for run_number, (_, meta_row) in enumerate(
    catalog_to_run.iterrows(),
    start=1
):
    path = Path(
        meta_row["path"]
    )

    print("\n" + "=" * 90)
    print(
        f"[{run_number}/{len(catalog_to_run)}] "
        f"{meta_row['folder']} / {path.name}"
    )
    print("=" * 90)

    try:
        row = run_one_dataset(
            path,
            meta_row
        )

        # Persist summary immediately.
        summary = update_summary_csv(
            [row]
        )

        print(
            f"Progress persisted: "
            f"{len(summary)} rows in summary.csv"
        )

    except Exception as exc:
        failure = {
            "folder": meta_row["folder"],
            "dataset": path.name,
            "error_type": type(exc).__name__,
            "error": str(exc),
        }

        failures.append(
            failure
        )

        print(
            f"ERROR: "
            f"{type(exc).__name__}: {exc}"
        )

        pd.DataFrame(
            failures
        ).to_csv(
            OUT_DIR / "batch_failures.csv",
            index=False
        )

        if not CONTINUE_ON_ERROR:
            raise


if SUMMARY_PATH.exists():
    summary = pd.read_csv(
        SUMMARY_PATH
    )
else:
    summary = pd.DataFrame()


print("\nRUN COMPLETE")
print("Summary:", SUMMARY_PATH)
print("Completed rows:", len(summary))
print("Failures:", len(failures))

display(summary)

if failures:
    display(
        pd.DataFrame(failures)
    )


## Optional summary plot

This is intentionally simple: total convergence-run time versus dataset size.

For a rigorous Atomic-vs-Reduction comparison, create the same summary fields in the reduction notebook and compare like-for-like runs using the same initial centroids.


In [ ]:
import matplotlib.pyplot as plt

if len(summary):
    plot_df = summary[
        summary["Implementation"]
        == "gpu_gemm_v3_aligned"
    ].copy()

    if len(plot_df):
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )

        for k, group in plot_df.groupby(
            "K"
        ):
            group = group.sort_values(
                "Rows"
            )

            ax.plot(
                group["Rows"],
                group["Execution time"],
                marker="o",
                label=f"K={k}"
            )

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Rows")
        ax.set_ylabel(
            "Timed convergence-run time (s)"
        )
        ax.set_title(
            "GPU GEMM K-Means V3 Runtime "
            "(JIT Warm-up Excluded)"
        )
        ax.grid(
            True,
            which="both",
            alpha=0.3
        )
        ax.legend()

        fig.tight_layout()

        plot_path = (
            OUT_DIR
            / "gemm_v3_total_runtime.png"
        )

        fig.savefig(
            plot_path,
            dpi=150
        )

        print(
            "Saved plot:",
            plot_path
        )

        plt.show()


## Cross-implementation comparison

Joins the four `summary.csv` files on dataset name. Before reading any timing, check:

- **same start:** `Initial centroid SHA256` must be identical across CPU / Reduction / GEMM,
- **same answer:** iterations should match and WCSS should agree to ~1e-9 relative
  (a different iteration count usually means one near-tie point flipped and the run took a slightly different path).

Then compare `Per iteration ms`. Total time also depends on the iteration count, so per-iteration time is the fair speed metric.

Set `CPU_RESULTS_DIR` to wherever your CPU V3 results live on Drive.


In [ ]:
GPU_RESULT_DIRS = {
    "atomic": DATA_ROOT / "results_gpu_atomic_v3",
    "reduction": DATA_ROOT / "results_gpu_reduction_v3_aligned",
    "gemm": OUT_DIR,
}
CPU_RESULTS_DIR = DATA_ROOT / "cpu_baseline_results"   # <- adjust to your folder


def load_gpu_summary(name, folder):
    path = Path(folder) / "summary.csv"
    if not path.exists():
        print(f"missing: {path}")
        return None
    s = pd.read_csv(path)
    keep = {
        "Dataset": "Dataset",
        "Rows": "Rows",
        "K": "K",
        "Per iteration ms": f"{name}_ms_per_iter",
        "Iterations": f"{name}_iters",
        "WCSS": f"{name}_wcss",
    }
    if "Initial centroid SHA256" in s.columns:
        keep["Initial centroid SHA256"] = f"{name}_sha"
    return s[list(keep)].rename(columns=keep)


def load_cpu_summary(folder):
    path = Path(folder) / "summary.csv"
    if not path.exists():
        print(f"missing: {path}")
        return None
    s = pd.read_csv(path)
    return s.rename(columns={
        "file": "Dataset",
        "per_iter_ms_median": "cpu_ms_per_iter",
        "conv_iterations": "cpu_iters",
        "WCSS": "cpu_wcss",
        "initial_centroid_sha256": "cpu_sha",
    })[["Dataset", "cpu_ms_per_iter", "cpu_iters", "cpu_wcss", "cpu_sha"]]


comparison = None
for name, folder in GPU_RESULT_DIRS.items():
    part = load_gpu_summary(name, folder)
    if part is None:
        continue
    if comparison is None:
        comparison = part
    else:
        comparison = comparison.merge(
            part.drop(columns=["Rows", "K"]), on="Dataset", how="outer"
        )

cpu_part = load_cpu_summary(CPU_RESULTS_DIR)
if comparison is not None and cpu_part is not None:
    comparison = comparison.merge(cpu_part, on="Dataset", how="left")

if comparison is not None:
    comparison = comparison.sort_values(["Rows", "K"]).reset_index(drop=True)

    sha_cols = [c for c in comparison.columns if c.endswith("_sha")]
    if len(sha_cols) > 1:
        comparison["same_start"] = comparison[sha_cols].nunique(axis=1, dropna=True) == 1

    if "gemm_wcss" in comparison and "atomic_wcss" in comparison:
        comparison["gemm_vs_atomic_wcss_rel"] = (
            (comparison["gemm_wcss"] - comparison["atomic_wcss"]).abs()
            / comparison["atomic_wcss"]
        )

    for other in ("cpu", "atomic", "reduction"):
        col = f"{other}_ms_per_iter"
        if col in comparison and "gemm_ms_per_iter" in comparison:
            comparison[f"gemm_speedup_vs_{other}"] = (
                comparison[col] / comparison["gemm_ms_per_iter"]
            )

    comparison.to_csv(OUT_DIR / "comparison_all_implementations.csv", index=False)
    display(comparison.drop(columns=sha_cols))


## Final convergence interpretation

The final assignment-stability rule is:

\[
\frac{\text{changed assignments}}{N}
\le 10^{-4}
\]

for **3 consecutive iterations**.

In percentage terms:

\[
10^{-4}\times100 = 0.01\%
\]

So at most 0.01% of points may change clusters during each of the final three iterations.

The four V2 diagnostic histories showed that this criterion is reachable even when exact zero assignment changes is not:

- 1M / K=3 reaches three consecutive qualifying iterations by about iteration 78.
- 5M / K=9 reaches them by about iteration 372.
- 10M / K=17 reaches them by about iteration 183.
- 10K / K=3 already meets the stricter centroid-movement tolerance at iteration 2.

For the final CPU-vs-Atomic-vs-Reduction-vs-GEMM comparison, use the **same convergence rule and initial centroids in every implementation**.


## For the report — questions this notebook should let you answer

1. **Arithmetic intensity.** GEMM #1 does `2·N·(D+1)·K` flops and reads `N·(D+1)` doubles;
   with D=9, K≤17 that's only ~2K/8 ≈ 4 flops/byte, so it is **memory-bound**, not compute-bound.
   Is GEMM still faster than the direct kernel? Why or why not?
2. **Materialisation cost.** The direct kernel never writes distances; GEMM writes an `N×K`
   score matrix and reads it back for `argmin`, then builds an `N×K` one-hot. Estimate the
   extra bytes per iteration for 10M × K=17 and compare against measured ms/iter.
3. **Redundant work in GEMM #2.** One-hot has 1 nonzero per row, so the GEMM does K× extra
   multiply-adds. Optional variant: `cupyx.scipy.sparse` CSR SpMM, or keep the Atomic update
   and only swap the assignment to GEMM (hybrid).
4. **Scaling with D and K.** GEMM's advantage grows with D·K. Optional experiment:
   synthetic data with D=64/128, K=64/256 to find the crossover vs Atomic.
5. **Precision.** What happens to label agreement if you skip mean-centring, or use float32?
